[Reference](https://medium.com/@pankaj_pandey/9eae1c43cea2$0)

In [1]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    scores = {}
    for ranking in ranked_lists:          # e.g. [bm25_ids, dense_ids]
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

fused = reciprocal_rank_fusion([bm25_results, dense_results])[:50]

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")
results = client.query_points(
    collection_name="docs",
    prefetch=[
        models.Prefetch(                        # dense: semantics
            query=models.Document(text=query, model="sentence-transformers/all-MiniLM-L6-v2"),
            using="dense", limit=50,
        ),
        models.Prefetch(                        # sparse: exact terms
            query=models.Document(text=query, model="Qdrant/bm25"),
            using="sparse", limit=50,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),   # fuse the two
    limit=10,
).points